In [ ]:
# Legacy Jumpstart setup used ms-fabric-cli. The REST-based cell below does not require it.
# %pip install ms-fabric-cli --quiet


# ✅ Prerequisites for Gateway Monitoring

This document outlines the required setup to enable the **Gateway Monitoring** solution, which collects and reports health and activity data from the **On-Premises Data Gateway**.

---

## 🧩 1. Gateway Monitoring Solution Overview

The Gateway Monitoring solution is composed of:
- A **PowerShell script** installed and executed on the **On-Premises Data Gateway**
- A **Fabric notebook** that generates the base configuration used by the script
- Secure authentication using a **service principal**

The PowerShell script is responsible for sending heartbeat and reporting information, while the notebook simplifies initial configuration.

---

## 🔐 2. Service Principal Requirements

A **service principal (SP)** is required to authenticate against Microsoft Fabric.

The service principal **must** meet all the following conditions:
- ✅ Have **Fabric Admin API access**
- ✅ Be a **Member of the Fabric workspace** used by the solution
- ✅ Be an **Administrator of the On-Premises Data Gateway**

Without these permissions, the monitoring solution will not be able to retrieve metadata or report gateway status.

---

## 🖥️ 3. Install the PowerShell Script on the On-Premises Data Gateway

- The **Gateway Monitoring PowerShell script** must be installed on the machine hosting the **On-Premises Data Gateway**.
- The script runs locally and is responsible for:
  - Sending **heartbeat signals**
  - Sending **gateway monitoring reports**
  - Respecting the configuration values generated by the notebook

Ensure the script runs under an identity that has access to the generated configuration file.

---

## 📓 4. Generate the Base Configuration Using the Notebook

Running the **current notebook** will automatically generate the **base configuration file** required by the PowerShell script.

This file defines how the Gateway Monitoring process behaves and how often data is sent.

---

## ⚙️ 5. Configure Gateway Monitoring Variables

After the configuration file is created, you must review and update the following variables as needed:

- **HeartbeatEnable**  
  Enable or disable the gateway heartbeat.

- **HeartbeatInterval**  
  Interval (in seconds) at which heartbeat signals are sent.

- **ReportSendInterval**  
  Interval (in seconds) at which monitoring reports are sent.

- **ReportRetention**  
  Retention period (in days) for monitoring reports.

- **TenantId**  
  Tenant ID of the service principal.

- **AppId**  
  Application (Client) ID of the service principal.

These values control both authentication and the operational behavior of the monitoring process.

---

## 🧠 Why This Setup Matters

This configuration ensures:
- ✅ Secure authentication using a service principal
- ✅ Continuous visibility into gateway health
- ✅ Configurable heartbeat and reporting intervals
- ✅ Centralized monitoring aligned with Fabric governance

---

## ✅ Final Checklist

Before enabling Gateway Monitoring, confirm the following:

- ✔️ Service principal has **Fabric Admin API access**
- ✔️ Service principal is **Workspace Member**
- ✔️ Service principal is **Gateway Admin**
- ✔️ PowerShell script is installed on the **On-Premises Data Gateway**
- ✔️ Notebook has been executed to generate the base config file
- ✔️ Gateway Monitoring variables are correctly configured

🚀 Once completed, the Gateway Monitoring solution will begin reporting gateway health and activity based on the configured intervals.

---

## 📘 Learn More

For additional details and official guidance on platform monitoring, visit:

👉 **https://aka.ms/fabric-platform-monitoring**

Look for the **Gateway** section for in-depth information specific to gateway monitoring and observability.


**本地 Fork 适配说明（2026-11-12）**：运行前先在参数区填入当前环境的 Tenant ID / App ID。此版本使用 Notebook Runtime Context 和 Fabric REST API，不依赖原版 Fabric CLI。导出的执行输出已清理；**提交到 Git 的这一版尚未重新执行验证**。

- 默认配置：Heartbeat 30 秒，Report polling 60 秒。
- 为兼容上下游，同时生成 `TennatId` 和 `TenantId`。
- **严禁提交**生成的 `builtin/config.json`，其中含 Eventstream 连接凭据。


In [ ]:
HeartbeatEnable = True
HeartbeatInterval = 30
ReportSendInterval = 60
ReportRetention = 10
TenatId = ""  # REQUIRED: Entra tenant ID
AppId = ""    # REQUIRED: App registration client ID

In [ ]:
# 修改 1：用 requests 直接调用 Fabric REST API，移除本单元对 fab 子进程的依赖。
# 已验证：同一个 Notebook token 直接请求 API 返回 200，而 fab 1.9.0 报 Invalid audience。
# 删除原 subprocess/yaml/sempy 导入及 fab_* 辅助函数：它们仅服务于旧 CLI 调用，
# 本 Notebook 的其他现有单元没有引用这些函数。第 1 单元的 CLI 安装暂时保留，但本单元不再需要它。
import json
from pathlib import Path
from urllib.parse import urlparse

import notebookutils
import requests

# 修改 2：REST API 按 displayName 查找资源，名称不包含 CLI 专用的 .Eventstream 后缀。
heartbeat_eventstream = "GatewayMonitoringHeartbeat"
report_eventstream = "GatewayMonitoringReports"

# 修改 3：继续使用 Notebook 当前运行身份，不另行登录或使用配置中的服务主体。
# 每次运行重新获取 token，避免重用上次执行留下的旧 token；不打印 token。
# workspace_id 来自当前 Notebook 上下文，不硬编码到其他工作区。
workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
token = notebookutils.credentials.getToken('pbi')
fabric_api_base = "https://api.fabric.microsoft.com/v1"
fabric_headers = {
    "Authorization": f"Bearer {token}",
    "x-ms-fabric-skill": "eventstream-cli",
}


def fabric_get_json(url):
    # 修改 4：请求设置超时，先检查 HTTP 状态，再解析 JSON。
    # 原 silently_continue=True 会吞掉 CLI 认证错误，最终变成误导性的 JSONDecodeError。
    # 限制分页 URL 的域名，避免把 Authorization 发送到其他服务；不自动跟随重定向。
    parsed_url = urlparse(url)
    if parsed_url.scheme != "https" or parsed_url.netloc != "api.fabric.microsoft.com":
        raise ValueError("拒绝向 Fabric API 以外的地址发送访问令牌。")
    try:
        response = requests.get(
            url, headers=fabric_headers, timeout=30, allow_redirects=False
        )
    except requests.RequestException:
        raise RuntimeError("Fabric API 请求失败或超时，请检查网络后重试。") from None
    if response.status_code != 200:
        # 不输出响应全文，尤其是连接信息接口的响应，以免泄露凭据。
        error_code = "Unknown"
        try:
            error_body = response.json()
            if isinstance(error_body, dict):
                error_code = error_body.get("errorCode", "Unknown")
        except ValueError:
            pass
        request_id = response.headers.get("requestId", "未提供")
        raise RuntimeError(
            f"Fabric API 失败：HTTP {response.status_code}；错误码 {error_code}；"
            f"requestId={request_id}。"
        )
    try:
        payload = response.json()
    except ValueError:
        raise RuntimeError("Fabric API 返回 HTTP 200，但响应不是有效 JSON。") from None
    if not isinstance(payload, dict):
        raise RuntimeError("Fabric API 响应结构异常：预期 JSON 对象。")
    return payload


# 修改 5：从工作区实际资源清单解析 ID，处理分页，并要求名称唯一。
# 原代码可能把 CLI 的报错文本当成 ID；现在找不到或重名时立即停止。
eventstreams = []
next_url = f"{fabric_api_base}/workspaces/{workspace_id}/items?type=Eventstream"
seen_urls = set()
while next_url:
    if next_url in seen_urls:
        raise RuntimeError("Fabric API 返回重复分页地址，停止以避免无限循环。")
    seen_urls.add(next_url)
    page = fabric_get_json(next_url)
    eventstreams.extend(page.get("value", []))
    next_url = page.get("continuationUri")


def get_eventstream_id(display_name):
    matches = [
        item for item in eventstreams
        if item.get("type") == "Eventstream" and item.get("displayName") == display_name
    ]
    if len(matches) != 1 or not matches[0].get("id"):
        raise RuntimeError(
            f"Eventstream '{display_name}' 必须唯一且具有有效 ID；找到 {len(matches)} 个。"
        )
    return matches[0]["id"]


def get_eventstream_connection(eventstream_id, display_name):
    endpoint = f"{fabric_api_base}/workspaces/{workspace_id}/eventstreams/{eventstream_id}"
    topology = fabric_get_json(f"{endpoint}/topology")
    # 修改 6：只选择支持连接信息接口的 CustomEndpoint 源，不再盲取 sources[0]。
    # 若不存在或有多个，则明确报错，避免拿错源。多个源时需后续指定源名称。
    sources = [
        source for source in topology.get("sources", [])
        if source.get("type") == "CustomEndpoint"
    ]
    if len(sources) != 1 or not sources[0].get("id"):
        raise RuntimeError(
            f"'{display_name}' 需要唯一的 CustomEndpoint 源；找到 {len(sources)} 个。"
        )
    source_id = sources[0]["id"]
    # 修改 7：直接 REST 响应没有 fab api 输出中的 text 包装层。
    # 连接接口需要 Eventstream.ReadWrite.All scope 及相应资源权限；
    # 清单查询成功并不保证此接口也获准。若返回 403，上面的 HTTP 检查会明确报告。
    # 此请求会读取访问密钥：仅供写入原配置文件，不输出连接串或整个响应。
    connection = fabric_get_json(f"{endpoint}/sources/{source_id}/connection")
    connection_string = connection.get("accessKeys", {}).get("primaryConnectionString")
    if not isinstance(connection_string, str) or not connection_string.strip():
        raise RuntimeError(f"'{display_name}' 未返回有效的 primaryConnectionString。")
    return connection_string


# 修改 8：先检查第 3 单元的参数是否已定义，提示执行顺序，避免新会话变量缺失。
required_parameters = (
    "HeartbeatEnable", "HeartbeatInterval", "ReportSendInterval",
    "ReportRetention", "TenatId", "AppId",
)
missing_parameters = [name for name in required_parameters if name not in globals()]
if missing_parameters:
    raise RuntimeError(f"请先运行第 3 单元，缺少参数：{', '.join(missing_parameters)}")

heartbeat_id = get_eventstream_id(heartbeat_eventstream)
report_id = get_eventstream_id(report_eventstream)
heartbeat_connection = get_eventstream_connection(heartbeat_id, heartbeat_eventstream)
report_connection = get_eventstream_connection(report_id, report_eventstream)

# 修改 9：保留原配置结构，并增加兼容的 TenantId 键，避免影响下游 PowerShell 脚本。
# 兼容原始 TennatId 拼写，同时添加下游实际使用的 TenantId。
# Windows 路径加 r 前缀，明确反斜杠为字面值；实际输出路径文本保持不变。
GatewayLogUploadConfig = {
    "GatewayId": "",
    "GatewayLogsPath": [r"C:\Windows\ServiceProfiles\PBIEgwService\AppData\Local\Microsoft\On-premises data gateway"],
    "RootPath": "raw",
    "OutputPath": r".\data",
    "HeartbeatEnable": HeartbeatEnable,
    "HeartbeatInterval": HeartbeatInterval,
    "ReportSendInterval": ReportSendInterval,
    "ReportRetention": ReportRetention,
    "VerboseLogSendInterval": 600,
    "ServicePrincipal": {
        "TennatId": TenatId,
        "TenantId": TenatId,  # Downstream PowerShell scripts read this correctly spelled key.
        "AppId": AppId,
        "SecretText": "",
    },
    "EventHubs": {
        "UploadReports": True,
        "ConnectionStrings": [
            {"Report": "Heartbeat", "EventHubConnectionString": heartbeat_connection},
            {"Report": "Reports", "EventHubConnectionString": report_connection},
        ],
    },
    "Lakehouse": {
        "UploadReports": False,
        "UploadLogs": False,
        "WorkspaceName": "",
        "LakehouseName": "",
    },
    "ConnectionProperties": {
        "MaximumRetryCount": 3,
        "RetryIntervalSec": 1,
    },
}

# 修改 10：使用 Fabric Notebook 资源的绝对路径，不依赖当前工作目录。
# with 自动关闭文件；全部 API 查询成功后才覆盖 config.json。
# 文件仍包含原方案要求的连接密钥，应按凭据文件管理；输出只显示生成结果。
config_path = Path(notebookutils.nbResPath) / "builtin" / "config.json"
config_path.parent.mkdir(parents=True, exist_ok=True)
with config_path.open("w", encoding="utf-8") as out_file:
    json.dump(GatewayLogUploadConfig, out_file, indent=6)
print("已生成 Notebook builtin/config.json；包含心跳与报告连接配置，密钥未显示。")